# Train Và So Sánh Baseline Cho AI-Generated Image Detection

Notebook này chạy trên Google Colab để huấn luyện và đánh giá các baseline cho bài toán phát hiện ảnh AI-generated.

Notebook hỗ trợ hai protocol:

- `cross_generator`: train trên một generator, test trên val của tất cả generator.
- `combined`: train trên toàn bộ split `train`, test trên split `val` hoặc `test`.

Notebook không tạo dữ liệu giả, không tự chia lại dữ liệu khi metadata đã có split gốc, và chỉ sinh kết quả sau khi model thực sự chạy.

## 0. Cài đặt phụ thuộc

Colab thường đã có PyTorch, torchvision, pandas, scikit-learn và matplotlib. Cell này chỉ cài thêm `open_clip_torch` khi bạn chọn `MODEL_NAME = "clip_linear_probe"` hoặc dùng `FEATURE_EXTRACTOR = "clip"`.

In [ ]:
import importlib.util
import subprocess
import sys


def install_if_missing(package_name: str, import_name: str | None = None) -> None:
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Đang cài {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
    else:
        print(f"Đã có {package_name}.")

## 1. Import thư viện và cấu hình

Không hard-code đường dẫn Windows trong Colab. Hãy chỉnh các biến `PROJECT_ROOT`, `DATA_ROOT`, `METADATA_ROOT`, `OUTPUT_ROOT`, `REPORT_ROOT` theo Google Drive của bạn.

Nếu bạn đã lưu output từ notebook chuẩn bị metadata bằng cell lưu Drive, mặc định metadata nằm ở:

```text
/content/drive/MyDrive/genimage_prepared_data/metadata
```

`DATA_ROOT` phải trỏ tới thư mục chứa ảnh thật sự. Nếu metadata lưu `image_path` từ runtime cũ và đường dẫn đó không còn tồn tại, notebook sẽ thử dựng lại đường dẫn bằng `DATA_ROOT / relative_path`.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")


PROJECT_ROOT = Path("/content/drive/MyDrive/genimage_baseline")
DATA_ROOT = Path("/content/drive/MyDrive/tiny-genimage")
METADATA_ROOT = Path("/content/drive/MyDrive/genimage_prepared_data/metadata")
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
REPORT_ROOT = PROJECT_ROOT / "report"

# Dùng cho EXPERIMENT_CASE = "cross_generator".
# Có thể đặt tên đầy đủ như "imagenet_ai_0419_biggan" hoặc tên ngắn như "biggan".
BASE_GENERATOR = "imagenet_ai_0419_biggan"

EXPERIMENT_CASE = "combined"  # "cross_generator" hoặc "combined"
MODEL_NAME = "resnet50"       # resnet50, swin_t, cnnspot, clip_linear_probe, fft, logistic_regression, linear_svm, random_forest, knn
FEATURE_EXTRACTOR = "fft"     # fft, resnet50, clip

BATCH_SIZE = 32
NUM_WORKERS = 2
NUM_EPOCHS = 3
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 224
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_PRETRAINED = True
RESUME_CHECKPOINT = None

EARLY_STOPPING_PATIENCE = 3
MAX_FEATURE_SAMPLES = None  # đặt số nguyên nếu muốn chạy thử nhanh, None để dùng toàn bộ
CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"
FFT_FEATURE_SIZE = 64

print("DEVICE:", DEVICE)
print("EXPERIMENT_CASE:", EXPERIMENT_CASE)
print("MODEL_NAME:", MODEL_NAME)
print("FEATURE_EXTRACTOR:", FEATURE_EXTRACTOR)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("METADATA_ROOT:", METADATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("REPORT_ROOT:", REPORT_ROOT)

## 3. Hàm tiện ích, seed và thư mục output

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def timestamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


def make_run_dir() -> Path:
    base_name = f"{EXPERIMENT_CASE}_{MODEL_NAME}_{timestamp()}"
    run_dir = OUTPUT_ROOT / EXPERIMENT_CASE / MODEL_NAME / base_name
    for sub in ["checkpoints", "features", "predictions", "metrics", "plots"]:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)
    return run_dir


def require_file(path: Path, message: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{message}: {path}")


def save_json(obj: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


set_seed(RANDOM_SEED)
RUN_DIR = make_run_dir()
print("RUN_DIR:", RUN_DIR)

## 4. Load và validate metadata

In [ ]:
REQUIRED_COLUMNS = ["image_path", "generator", "original_split", "label", "label_name"]


def resolve_image_path(row: pd.Series) -> str:
    raw_path = Path(str(row["image_path"]))
    if raw_path.exists():
        return str(raw_path)
    if "relative_path" in row and pd.notna(row["relative_path"]):
        candidate = DATA_ROOT / str(row["relative_path"])
        if candidate.exists():
            return str(candidate)
    return str(raw_path)


def read_metadata_csv(path: Path) -> pd.DataFrame:
    require_file(path, "Không tìm thấy file metadata")
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Metadata thiếu cột bắt buộc {missing}: {path}")
    df = df.copy()
    df["image_path"] = df.apply(resolve_image_path, axis=1)
    df["label"] = df["label"].astype(int)
    df["generator"] = df["generator"].astype(str)
    df["original_split"] = df["original_split"].astype(str)
    return df


def resolve_base_generator(base_generator: str, generators: List[str]) -> str:
    if base_generator in generators:
        return base_generator
    query = str(base_generator).lower().strip()
    matches = [g for g in generators if query == g.lower() or query in g.lower()]
    if len(matches) == 1:
        print(f"BASE_GENERATOR={base_generator!r} được ánh xạ thành {matches[0]!r}.")
        return matches[0]
    raise ValueError(
        f"BASE_GENERATOR={base_generator!r} không khớp duy nhất. "
        f"Các generator hợp lệ: {generators}"
    )


def load_metadata() -> Tuple[pd.DataFrame, pd.DataFrame]:
    combined_path = METADATA_ROOT / "combined" / "all_generators_metadata.csv"
    combined_df = read_metadata_csv(combined_path)

    generators = sorted(combined_df["generator"].unique().tolist())

    if EXPERIMENT_CASE == "combined":
        train_df = combined_df[combined_df["original_split"] == "train"].copy()
        test_df = combined_df[combined_df["original_split"].isin(["val", "test"])].copy()
    elif EXPERIMENT_CASE == "cross_generator":
        base_generator = resolve_base_generator(BASE_GENERATOR, generators)
        train_df = combined_df[
            (combined_df["generator"] == base_generator)
            & (combined_df["original_split"] == "train")
        ].copy()
        test_df = combined_df[combined_df["original_split"].isin(["val", "test"])].copy()
    else:
        raise ValueError('EXPERIMENT_CASE phải là "cross_generator" hoặc "combined".')

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    return train_df, test_df


def validate_metadata(train_df: pd.DataFrame, test_df: pd.DataFrame) -> Dict[str, Any]:
    issues: Dict[str, Any] = {}

    for name, df in [("train", train_df), ("test", test_df)]:
        if df.empty:
            raise ValueError(f"Metadata {name} bị rỗng.")
        invalid_labels = sorted(set(df["label"].unique()) - {0, 1})
        if invalid_labels:
            raise ValueError(f"Metadata {name} có label không hợp lệ: {invalid_labels}")
        duplicate_count = int(df.duplicated().sum())
        issues[f"{name}_duplicate_rows"] = duplicate_count
        missing_paths = df[~df["image_path"].map(lambda x: Path(str(x)).exists())]
        issues[f"{name}_missing_images"] = int(len(missing_paths))
        if len(missing_paths) > 0:
            display(missing_paths.head())
            raise FileNotFoundError(
                f"Có {len(missing_paths)} ảnh không tồn tại trong metadata {name}. "
                "Hãy kiểm tra DATA_ROOT hoặc image_path/relative_path."
            )

    train_paths = set(train_df["image_path"].astype(str))
    test_paths = set(test_df["image_path"].astype(str))
    overlap_paths = train_paths & test_paths
    issues["image_path_overlap"] = int(len(overlap_paths))

    if "sha256" in train_df.columns and "sha256" in test_df.columns:
        train_sha = set(train_df["sha256"].dropna().astype(str))
        test_sha = set(test_df["sha256"].dropna().astype(str))
        issues["sha256_overlap"] = int(len(train_sha & test_sha))
    else:
        issues["sha256_overlap"] = None

    if EXPERIMENT_CASE == "cross_generator":
        base_generator = resolve_base_generator(BASE_GENERATOR, sorted(pd.concat([train_df, test_df])["generator"].unique()))
        leaked_fake = train_df[(train_df["generator"] != base_generator) & (train_df["label"] == 1)]
        if not leaked_fake.empty:
            raise AssertionError("Train cross-generator chứa fake của generator khác.")

    return issues


train_df, test_df = load_metadata()
validation_issues = validate_metadata(train_df, test_df)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train generators:", sorted(train_df["generator"].unique().tolist()))
print("Test generators:", sorted(test_df["generator"].unique().tolist()))
print("Validation:", validation_issues)

## 5. Dataset, transform và augmentation

In [ ]:
class JpegBlurAugment:
    def __init__(self, jpeg_prob: float = 0.5, blur_prob: float = 0.5, quality_range: Tuple[int, int] = (30, 100)):
        self.jpeg_prob = jpeg_prob
        self.blur_prob = blur_prob
        self.quality_range = quality_range

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() < self.blur_prob:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.0, 3.0)))
        if random.random() < self.jpeg_prob:
            import io
            buffer = io.BytesIO()
            quality = random.randint(*self.quality_range)
            img.save(buffer, format="JPEG", quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).convert("RGB")
        return img


def get_transforms(model_name: str, train: bool) -> transforms.Compose:
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ops: List[Callable] = []
    if train:
        if model_name == "cnnspot":
            ops.append(JpegBlurAugment())
        ops.extend([
            transforms.RandomResizedCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
        ])
    else:
        ops.extend([
            transforms.Resize(int(IMAGE_SIZE * 1.15)),
            transforms.CenterCrop(IMAGE_SIZE),
        ])
    ops.extend([transforms.ToTensor(), normalize])
    return transforms.Compose(ops)


class ImageMetadataDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Optional[Callable] = None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        meta = {
            "image_path": row["image_path"],
            "generator": row["generator"],
            "label_name": row["label_name"],
        }
        return img, label, meta


def collate_batch(batch: List[Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]]):
    images, labels, metas = zip(*batch)
    return torch.stack(images), torch.stack(labels), list(metas)


def build_dataloader(df: pd.DataFrame, train: bool, model_name: str) -> DataLoader:
    dataset = ImageMetadataDataset(df, transform=get_transforms(model_name, train=train))
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_batch,
    )

## 6. Model deep learning

In [ ]:
def build_model(model_name: str) -> nn.Module:
    model_name = model_name.lower()
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if USE_PRETRAINED else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, 2)
        return model

    if model_name == "swin_t":
        weights = models.Swin_T_Weights.IMAGENET1K_V1 if USE_PRETRAINED else None
        model = models.swin_t(weights=weights)
        model.head = nn.Linear(model.head.in_features, 2)
        return model

    if model_name == "cnnspot":
        # Bản tái hiện baseline CNNSpot: ResNet-50 kết hợp augmentation JPEG compression và Gaussian blur.
        # Không gọi ResNet-50 thường là CNNSpot nếu thiếu augmentation tương ứng.
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if USE_PRETRAINED else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, 2)
        return model

    raise ValueError(f"MODEL_NAME không phải deep model được hỗ trợ: {model_name}")


def is_deep_model(model_name: str) -> bool:
    return model_name in {"resnet50", "swin_t", "cnnspot"}

## 7. Metric, prediction và plot

In [ ]:
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, Any]:
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        metrics["average_precision"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics


def evaluate_by_generator(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for generator, part in pred_df.groupby("generator"):
        metrics = compute_metrics(part["label"].to_numpy(), part["fake_probability"].to_numpy())
        metrics["generator"] = generator
        rows.append(metrics)
    return pd.DataFrame(rows)


def save_predictions(pred_df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pred_df.to_csv(path, index=False)


def plot_confusion_matrix_from_metrics(metrics: Dict[str, Any], path: Path) -> None:
    cm = np.array(metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_roc_curve(y_true: np.ndarray, y_prob: np.ndarray, path: Path) -> None:
    if len(np.unique(y_true)) < 2:
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr)
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_generator_comparison(generator_metrics: pd.DataFrame, path: Path) -> None:
    if generator_metrics.empty or "balanced_accuracy" not in generator_metrics.columns:
        return
    df = generator_metrics.sort_values("generator")
    fig, ax = plt.subplots(figsize=(max(6, len(df) * 1.2), 4))
    ax.bar(df["generator"], df["balanced_accuracy"])
    ax.set_ylabel("Balanced Accuracy")
    ax.set_xticklabels(df["generator"], rotation=45, ha="right")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

## 8. Train và evaluate deep model

In [ ]:
def run_inference_deep(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    probs, labels, paths, generators = [], [], [], []
    with torch.no_grad():
        for images, y, metas in tqdm(loader, desc="Inference"):
            images = images.to(DEVICE)
            logits = model(images)
            batch_prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            probs.extend(batch_prob.tolist())
            labels.extend(y.numpy().tolist())
            paths.extend([m["image_path"] for m in metas])
            generators.extend([m["generator"] for m in metas])
    return pd.DataFrame({
        "image_path": paths,
        "generator": generators,
        "label": labels,
        "fake_probability": probs,
    })


def train_deep_model() -> Tuple[pd.DataFrame, Dict[str, Any]]:
    train_loader = build_dataloader(train_df, train=True, model_name=MODEL_NAME)
    test_loader = build_dataloader(test_df, train=False, model_name=MODEL_NAME)

    model = build_model(MODEL_NAME).to(DEVICE)
    if RESUME_CHECKPOINT:
        checkpoint = torch.load(RESUME_CHECKPOINT, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(NUM_EPOCHS, 1))
    scaler = GradScaler(enabled=(DEVICE == "cuda"))

    best_score = -math.inf
    patience = 0
    best_path = RUN_DIR / "checkpoints" / "best_model.pt"

    for epoch in range(NUM_EPOCHS):
        model.train()
        losses = []
        for images, y, _ in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"):
            images = images.to(DEVICE)
            y = y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(DEVICE == "cuda")):
                logits = model(images)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        scheduler.step()

        pred_df = run_inference_deep(model, test_loader)
        metrics = compute_metrics(pred_df["label"].to_numpy(), pred_df["fake_probability"].to_numpy())
        score = metrics["balanced_accuracy"]
        print(f"Epoch {epoch + 1}: loss={np.mean(losses):.4f}, balanced_accuracy={score:.4f}")

        if score > best_score:
            best_score = score
            patience = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "model_name": MODEL_NAME,
                "epoch": epoch,
                "metrics": metrics,
            }, best_path)
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print("Early stopping.")
                break

    checkpoint = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    pred_df = run_inference_deep(model, test_loader)
    pred_df["predicted_label"] = (pred_df["fake_probability"] >= 0.5).astype(int)
    return pred_df, checkpoint["metrics"]

## 9. Feature extraction và traditional ML

In [ ]:
def fft_feature_from_image(path: str) -> np.ndarray:
    img = Image.open(path).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    spectrum = np.fft.fftshift(np.fft.fft2(arr))
    magnitude = np.log1p(np.abs(spectrum))
    center = magnitude.shape[0] // 2
    radius = FFT_FEATURE_SIZE // 2
    crop = magnitude[center - radius:center + radius, center - radius:center + radius]
    return crop.flatten().astype(np.float32)


def build_resnet_feature_extractor() -> Tuple[nn.Module, Callable]:
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if USE_PRETRAINED else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Identity()
    model = model.to(DEVICE).eval()
    return model, get_transforms("resnet50", train=False)


def build_clip_feature_extractor() -> Tuple[Any, Callable]:
    install_if_missing("open_clip_torch", "open_clip")
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME,
        pretrained=CLIP_PRETRAINED,
        device=DEVICE,
    )
    model.eval()
    return model, preprocess


def feature_cache_path(df_name: str, extractor: str) -> Path:
    return RUN_DIR / "features" / f"{df_name}_{extractor}_features.npz"


def extract_features(df: pd.DataFrame, df_name: str, extractor: str) -> Tuple[np.ndarray, np.ndarray]:
    cache_path = feature_cache_path(df_name, extractor)
    if cache_path.exists():
        print("Đọc feature cache:", cache_path)
        data = np.load(cache_path)
        return data["features"], data["labels"]

    work_df = df.copy()
    if MAX_FEATURE_SAMPLES is not None:
        work_df = work_df.sample(n=min(MAX_FEATURE_SAMPLES, len(work_df)), random_state=RANDOM_SEED)

    labels = work_df["label"].to_numpy().astype(int)

    if extractor == "fft":
        features = np.stack([fft_feature_from_image(path) for path in tqdm(work_df["image_path"], desc=f"FFT {df_name}")])
    elif extractor == "resnet50":
        model, preprocess = build_resnet_feature_extractor()
        features_list = []
        dataset = ImageMetadataDataset(work_df, transform=preprocess)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_batch)
        with torch.no_grad():
            for images, _, _ in tqdm(loader, desc=f"ResNet features {df_name}"):
                feats = model(images.to(DEVICE)).detach().cpu().numpy()
                features_list.append(feats)
        features = np.concatenate(features_list, axis=0)
    elif extractor == "clip":
        model, preprocess = build_clip_feature_extractor()
        features_list = []
        dataset = ImageMetadataDataset(work_df, transform=preprocess)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_batch)
        with torch.no_grad():
            for images, _, _ in tqdm(loader, desc=f"CLIP features {df_name}"):
                feats = model.encode_image(images.to(DEVICE))
                feats = feats / feats.norm(dim=-1, keepdim=True)
                features_list.append(feats.detach().cpu().numpy())
        features = np.concatenate(features_list, axis=0)
    else:
        raise ValueError(f"FEATURE_EXTRACTOR không hợp lệ: {extractor}")

    np.savez_compressed(cache_path, features=features, labels=labels)
    print("Đã lưu feature cache:", cache_path)
    return features, labels


def build_ml_classifier(model_name: str):
    if model_name == "logistic_regression":
        return Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    if model_name == "linear_svm":
        return Pipeline([("scaler", StandardScaler()), ("clf", LinearSVC(class_weight="balanced"))])
    if model_name == "random_forest":
        return RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, class_weight="balanced", n_jobs=-1)
    if model_name == "knn":
        return Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=5))])
    if model_name in {"fft", "clip_linear_probe"}:
        return Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    raise ValueError(f"ML classifier không hợp lệ: {model_name}")


def predict_probability_ml(clf: Any, x: np.ndarray) -> np.ndarray:
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(x)[:, 1]
    scores = clf.decision_function(x)
    return 1.0 / (1.0 + np.exp(-scores))


def train_ml_classifier() -> Tuple[pd.DataFrame, Dict[str, Any]]:
    extractor = FEATURE_EXTRACTOR
    if MODEL_NAME == "fft":
        extractor = "fft"
    elif MODEL_NAME == "clip_linear_probe":
        extractor = "clip"

    x_train, y_train = extract_features(train_df, "train", extractor)
    x_test, y_test = extract_features(test_df, "test", extractor)

    clf = build_ml_classifier(MODEL_NAME)
    clf.fit(x_train, y_train)
    joblib_path = RUN_DIR / "checkpoints" / f"{MODEL_NAME}.joblib"
    joblib.dump(clf, joblib_path)
    print("Đã lưu model:", joblib_path)

    y_prob = predict_probability_ml(clf, x_test)
    pred_df = test_df[["image_path", "generator", "label"]].copy().reset_index(drop=True)
    pred_df["fake_probability"] = y_prob
    pred_df["predicted_label"] = (pred_df["fake_probability"] >= 0.5).astype(int)
    metrics = compute_metrics(y_test, y_prob)
    return pred_df, metrics

## 10. Chạy model được chọn và lưu kết quả

In [ ]:
if MODEL_NAME in {"clip_linear_probe", "fft", "logistic_regression", "linear_svm", "random_forest", "knn"}:
    predictions_df, overall_metrics = train_ml_classifier()
elif is_deep_model(MODEL_NAME):
    predictions_df, overall_metrics = train_deep_model()
else:
    raise ValueError(f"MODEL_NAME không được hỗ trợ: {MODEL_NAME}")

predictions_df["experiment_case"] = EXPERIMENT_CASE
predictions_df["model_name"] = MODEL_NAME
predictions_df["base_generator"] = BASE_GENERATOR if EXPERIMENT_CASE == "cross_generator" else "ALL"

generator_metrics_df = evaluate_by_generator(predictions_df)

pred_path = RUN_DIR / "predictions" / "predictions.csv"
overall_metrics_path = RUN_DIR / "metrics" / "overall_metrics.json"
generator_metrics_path = RUN_DIR / "metrics" / "generator_metrics.csv"

save_predictions(predictions_df, pred_path)
save_json(overall_metrics, overall_metrics_path)
generator_metrics_df.to_csv(generator_metrics_path, index=False)

plot_confusion_matrix_from_metrics(overall_metrics, RUN_DIR / "plots" / "confusion_matrix.png")
plot_roc_curve(predictions_df["label"].to_numpy(), predictions_df["fake_probability"].to_numpy(), RUN_DIR / "plots" / "roc_curve.png")
plot_generator_comparison(generator_metrics_df, RUN_DIR / "plots" / "generator_comparison.png")

print("Overall metrics:")
print(json.dumps(overall_metrics, indent=2, ensure_ascii=False))
print("Đã lưu predictions:", pred_path)
print("Đã lưu metrics:", overall_metrics_path)
display(generator_metrics_df)

## 11. Tổng hợp kết quả từ các lần chạy trước

In [ ]:
def collect_previous_metrics(outputs_root: Path) -> pd.DataFrame:
    rows = []
    for metrics_path in outputs_root.rglob("overall_metrics.json"):
        try:
            with metrics_path.open("r", encoding="utf-8") as f:
                metrics = json.load(f)
            run_dir = metrics_path.parents[1]
            parts = run_dir.relative_to(outputs_root).parts
            if len(parts) >= 3:
                experiment, model, run_name = parts[0], parts[1], parts[2]
            else:
                experiment, model, run_name = "unknown", "unknown", run_dir.name
            rows.append({
                "Experiment": experiment,
                "Model": model,
                "Run": run_name,
                "Base generator": BASE_GENERATOR if experiment == "cross_generator" else "ALL",
                "Accuracy": metrics.get("accuracy"),
                "Balanced Accuracy": metrics.get("balanced_accuracy"),
                "F1": metrics.get("f1"),
                "ROC-AUC": metrics.get("roc_auc"),
                "AP": metrics.get("average_precision"),
            })
        except Exception as exc:
            print("Bỏ qua metric lỗi:", metrics_path, exc)
    return pd.DataFrame(rows)


comparison_df = collect_previous_metrics(OUTPUT_ROOT)
comparison_path = OUTPUT_ROOT / "model_comparison_summary.csv"
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)

if not comparison_df.empty:
    for metric in ["Accuracy", "Balanced Accuracy", "F1", "ROC-AUC"]:
        plot_df = comparison_df.dropna(subset=[metric])
        if plot_df.empty:
            continue
        fig, ax = plt.subplots(figsize=(max(6, len(plot_df) * 1.2), 4))
        labels = plot_df["Experiment"] + "/" + plot_df["Model"]
        ax.bar(labels, plot_df[metric])
        ax.set_ylabel(metric)
        ax.set_xticklabels(labels, rotation=45, ha="right")
        fig.tight_layout()
        out_path = OUTPUT_ROOT / f"comparison_{metric.lower().replace(' ', '_')}.png"
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print("Đã lưu biểu đồ:", out_path)
else:
    print("Chưa có kết quả thực tế để tổng hợp ngoài lần chạy hiện tại.")

## 12. Sinh báo cáo Markdown

In [ ]:
def df_to_markdown(df: pd.DataFrame) -> str:
    if df.empty:
        return "Chưa có kết quả."
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(lines)


report_path = REPORT_ROOT / "baseline_model_comparison.md"
report = f'''# Baseline Model Comparison

## 1. Mục tiêu

Huấn luyện và đánh giá baseline cho bài toán phát hiện ảnh AI-generated.

## 2. Dataset và metadata

Metadata root:

```text
{METADATA_ROOT}
```

Train rows: {len(train_df)}

Test rows: {len(test_df)}

Validation:

```json
{json.dumps(validation_issues, ensure_ascii=False, indent=2)}
```

## 3. Hai protocol thí nghiệm

- TH1 `cross_generator`: train trên một generator, test trên val/test của tất cả generator.
- TH2 `combined`: train trên toàn bộ split train, test trên split val/test.

Protocol lần chạy này: `{EXPERIMENT_CASE}`.

## 4. Các baseline model

Hỗ trợ: ResNet-50, Swin-T, CNNSpot reproduction, CLIP Linear Probe, Spec/FFT baseline, Logistic Regression, Linear SVM, Random Forest, KNN.

## 5. Cấu hình huấn luyện

```json
{json.dumps({
    "model_name": MODEL_NAME,
    "feature_extractor": FEATURE_EXTRACTOR,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "image_size": IMAGE_SIZE,
    "use_pretrained": USE_PRETRAINED,
    "device": DEVICE,
}, ensure_ascii=False, indent=2)}
```

## 6. Feature dùng cho ML classifier

Feature extractor được chọn: `{FEATURE_EXTRACTOR}`. Với `clip_linear_probe`, notebook dùng CLIP frozen image encoder. Với `fft`, notebook dùng log magnitude spectrum crop.

## 7. Kết quả TH1

{"Kết quả nằm trong lần chạy này." if EXPERIMENT_CASE == "cross_generator" else "Chưa có kết quả trong lần chạy này."}

## 8. Kết quả TH2

{"Kết quả nằm trong lần chạy này." if EXPERIMENT_CASE == "combined" else "Chưa có kết quả trong lần chạy này."}

## 9. Kết quả theo generator

{df_to_markdown(generator_metrics_df)}

## 10. So sánh model

{df_to_markdown(comparison_df)}

## 11. Data leakage và validation

Notebook kiểm tra overlap theo `image_path` và theo `sha256` nếu metadata có cột `sha256`.

## 12. Hạn chế

Kết quả chỉ phản ánh các model đã thật sự chạy. Các model chưa chạy được ghi là chưa có kết quả, không tạo số liệu giả.

## 13. Kết luận

Notebook đã sinh prediction, metric, plot và báo cáo từ kết quả thực tế của lần chạy hiện tại.
'''

report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report, encoding="utf-8")
print("Đã sinh báo cáo:", report_path)